# TP3 : Fine-Tuning d'un LLM avec LoRA (float16) sur GPU T4
## Spécialisation sur le RGPD — Règlement (UE) 2016/679

---

## Objectifs du TP

À la fin de ce TP, vous serez capables de :
1. **Comprendre** les différences entre Full Fine-Tuning, LoRA et QLoRA
2. **Charger** un LLM Qwen3-4B en float16 sur un GPU T4 (15 Go VRAM)
3. **Configurer** les adaptateurs LoRA et comprendre chaque hyperparamètre
4. **Préparer** un dataset à partir d'un texte réglementaire (RGPD)
5. **Entraîner** le modèle à répondre en français sur le droit des données personnelles
6. **Visualiser** la courbe de loss et interpréter le surapprentissage
7. **Évaluer** et comparer le modèle avant/après fine-tuning
8. **Sauvegarder** et exporter les adaptateurs entraînés (GGUF / Ollama)

---

## Corpus utilisé dans ce TP

Nous allons fine-tuner le modèle sur le **Règlement Général sur la Protection des Données** :

| Source | Description |
|---|---|
| `CELEX_32016R0679_FR_TXT.pdf` | Règlement (UE) 2016/679 — texte officiel français (173 considérants + 99 articles) |
| `dataset_rgpd_fr.json` | 95 paires Q/R en français construites à partir du texte du RGPD |

> **Objectif pédagogique :** Après le fine-tuning, le modèle doit répondre **en français**
> de façon **structurée et précise** sur les concepts du RGPD (bases légales, droits des
> personnes, obligations des responsables, sanctions), même à des questions hors dataset.

---

## Architecture LoRA

```
  Entrée (x)
     ├──────────────────────────┐
     v                          v
  W (gelé, float16)        A (r×k) — init aléatoire
                            B (d×r) — init à zéro
                            r = 16 << d
     └──────────────────────────┘
  h = W·x + (alpha/r)·B·A·x

  Full FT  : entraîne W entier          → ~4B params
  LoRA     : gèle W, entraîne A et B   → ~1.6% des params
```

---

**Durée estimée** : ~2h  
**Niveau** : Master 2  
**Plateforme** : Google Colab (GPU T4)  
**Prérequis** : Uploadez `dataset_rgpd_fr.json` dans Google Drive · Activez le GPU T4


---
# Partie 1 : Vérification du GPU et Installation (10 min)

Avant toute chose, vérifions que l'A100 est bien activé. Il offre **80 Go de VRAM** — largement suffisant pour LoRA bf16 sur un modèle 27B (~56 Go requis).

Nous utilisons **Unsloth**, une bibliothèque optimisée qui accélère l'entraînement de **1.5× à 2×** et réduit la mémoire de **50%** par rapport aux approches classiques.

In [1]:
# Fix bitsandbytes pour Colab T4

# Sur T4 Colab, bitsandbytes a besoin de libnvJitLink.so.13
# Ce fix le trouve et crée le symlink AVANT d'importer unsloth.
# Si introuvable → on évite bitsandbytes (load_in_4bit=False).

import subprocess, os, glob

# 1. Chercher libnvJitLink sur tout le système
result = subprocess.run(
    ['find', '/usr', '/lib', '/opt', '-name', 'libnvJitLink.so*', '-type', 'f'],
    capture_output=True, text=True, timeout=30
)
lib_paths = [p for p in result.stdout.strip().split('\n') if p]
print(f'libnvJitLink trouvé : {lib_paths}')

# 2. Créer le symlink vers libnvJitLink.so.13
target = '/usr/lib/x86_64-linux-gnu/libnvJitLink.so.13'
if not os.path.exists(target) and lib_paths:
    try:
        os.symlink(lib_paths[0], target)
        print(f'✓ Symlink créé : {lib_paths[0]} → {target}')
    except Exception as e:
        print(f'⚠ Symlink échoué : {e}')
elif os.path.exists(target):
    print(f'✓ {target} existe déjà')
else:
    print('⚠ libnvJitLink introuvable — on utilisera load_in_16bit=True (pas de bitsandbytes)')

# 3. Mettre à jour LD_LIBRARY_PATH
for cuda_path in ['/usr/local/cuda/lib64', '/usr/local/cuda-13.0/lib64']:
    if os.path.exists(cuda_path):
        os.environ['LD_LIBRARY_PATH'] = cuda_path + ':' + os.environ.get('LD_LIBRARY_PATH', '')

os.environ['BITSANDBYTES_NOWELCOME'] = '1'
print('✓ Fix bitsandbytes appliqué')


In [2]:
# Vérification du GPU

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Aucun GPU détecté !\n"
        "→ Allez dans : Exécution > Modifier le type d'exécution > A100 GPU\n"
        "→ Puis relancez cette cellule."
    )

gpu_name   = torch.cuda.get_device_name(0)
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
compute    = torch.cuda.get_device_capability()

print(f" GPU détecté : {gpu_name}")
print(f"   VRAM totale          : {vram_total:.1f} Go")
print(f"   Compute Capability   : {compute[0]}.{compute[1]}")
print(f"   CUDA version         : {torch.version.cuda}")
print(f"   PyTorch version      : {torch.__version__}")

if vram_total < 50:
    print(f"\n  ATTENTION : VRAM insuffisante pour Qwen3.5-27B en LoRA bf16")
    print(f"   → Sur T4 (15 Go) : utilisez Qwen3.5-4B + load_in_4bit=True")
else:
    marge = vram_total - 56
    print(f"\n  VRAM suffisante pour Qwen3 -4B LoRA bf16")
    print(f"   → Requis : ~56 Go | Disponible : {vram_total:.0f} Go | Marge : +{marge:.0f} Go")

In [ ]:
# Installation d'Unsloth

# L'installation prend environ 2-3 minutes.
#
# IMPORTANT : Redémarrez le runtime après l'installation
# (Exécution > Redémarrer la session) puis relancez depuis la cellule 1.

#%%capture install_output
!pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo

print(" Installation terminée ! Redémarrez le runtime si c'est la première exécution.")


In [ ]:
!pip install -q bitsandbytes --upgrade


import os


os.environ["BITSANDBYTES_NOWELCOME"] = "1"


In [3]:
# Imports, seed global et suppression des warnings


import warnings
import random
import numpy as np
import torch

# Supprimer les FutureWarnings connus de Transformers v5
warnings.filterwarnings("ignore", category=FutureWarning, module="transformers")
warnings.filterwarnings("ignore", message=".*max_new_tokens.*max_length.*")

# Seed global — garantit que tous les étudiants obtiennent les mêmes résultats
GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)

from unsloth import FastLanguageModel
from unsloth import is_bfloat16_supported

print(f" Imports OK | Seed global : {GLOBAL_SEED}")
print(f" bfloat16 supporté : {is_bfloat16_supported()} (A100 = True) (T4 = False)")
print(f" Unsloth prêt.")

---
# Partie 2 : Chargement du modèle (15 min)

## Choix du modèle selon le GPU disponible

| Modèle | VRAM LoRA bf16 | VRAM QLoRA 4-bit | GPU recommandé |
|---|---|---|---|
| Qwen3.5-0.8B | 3 Go | 1.5 Go | T4, RTX 3060 |
| Qwen3.5-4B | 10 Go | 5 Go | T4, RTX 3080 |
| Qwen3.5-9B | 22 Go | 11 Go | RTX 4090, A10 |
| **Qwen3.5-27B** | **56 Go** | **28 Go** | **A100 80Go ← Notre cas** |
| Qwen3.5-35B-A3B | ~74 Go | ~37 Go | A100 80Go (marge serrée) |

## Pourquoi LoRA bf16 et pas QLoRA 4-bit ici ?

Unsloth **déconseille officiellement** QLoRA 4-bit pour Qwen3.5 (différences de quantification plus élevées que la normale). Avec 80 Go de VRAM, on charge le 27B en bf16 → **meilleure qualité**, pas de risque lié à NF4.

Sur GPU limité (T4, etc.) → `load_in_4bit=True` reste la seule option viable.

In [4]:
#  Chargement du modèle Qwen3-4B (LoRA float16)


#  Sur A100 80Go  → MODEL_NAME = "unsloth/Qwen3-14B"
#  Sur RTX 4090   → MODEL_NAME = "unsloth/Qwen3-8B"
#  Sur T4 (Colab) → MODEL_NAME = "unsloth/Qwen3-4B"  + load_in_16bit=True
#
#  NOTE T4 : load_in_4bit=False car bitsandbytes (libnvJitLink.so.13)
#  est absent sur Colab CUDA 13.x. On utilise float16 natif.
#  Qwen3-4B en float16 = ~8 Go → marge suffisante sur T4 (15 Go).

MODEL_NAME     = "unsloth/Qwen3-4B"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = torch.float16,   # Explicite float16 pour T4
    load_in_4bit    = False,           # Désactivé : bitsandbytes non dispo sur T4
    load_in_16bit   = True,            # float16 natif — recommandé Qwen3 sur T4
    full_finetuning = False,
)

total_params = sum(p.numel() for p in model.parameters())
mem_used     = torch.cuda.memory_allocated() / 1e9
vram_total   = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"\n{'=' * 60}")
print(f" Modèle chargé : {MODEL_NAME}")
print(f"   Paramètres totaux    : {total_params:,}")
print(f"   VRAM utilisée        : {mem_used:.2f} Go")
print(f"   VRAM disponible      : {vram_total:.0f} Go  →  marge : {vram_total - mem_used:.1f} Go")
print(f"   Longueur max séquence: {MAX_SEQ_LENGTH}")
print(f"   Précision            : float16 (recommandé pour T4)")
print(f"{'=' * 60}")


---
# Partie 3 : Configuration des Adaptateurs LoRA (20 min)

## Les hyperparamètres clés de LoRA

| Paramètre | Rôle | Valeur typique |
|---|---|---|
| **r** (rank) | Dimension des matrices A et B. Plus r est grand, plus le modèle a de capacité d'adaptation. | 8 à 64 |
| **lora_alpha** | Facteur d'échelle. `effective_lr = lr × (alpha/r)`. Souvent = r. | r à 2×r |
| **lora_dropout** | Régularisation : % d'activations à zéro. Recommandé = 0 (Unsloth). | 0 |
| **target_modules** | Quelles couches du Transformer reçoivent des adaptateurs LoRA. | Attention + MLP |

### Ce que dit le papier LoRA (Hu et al., 2021)

> *"Even r=1 suffices for many tasks on GPT-3"* — l'article montre que le rang r a peu d'impact une fois qu'on cible toutes les couches (Q, K, V, O, FFN). Ce qui compte davantage : **cibler toutes les matrices**, pas seulement Q et V.

```
Transformer Block
├── Multi-Head Attention
│   ├── q_proj  ← LoRA
│   ├── k_proj  ← LoRA
│   ├── v_proj  ← LoRA
│   └── o_proj  ← LoRA
└── MLP (Feed-Forward)
    ├── gate_proj ← LoRA
    ├── up_proj   ← LoRA
    └── down_proj ← LoRA
```

In [5]:
# Application des adaptateurs LoRA

# EXERCICE : Après un premier entraînement, modifiez ces valeurs
# et comparez l'impact sur la loss et les réponses :
#   r=4  vs r=16 vs r=64  → Capacité d'adaptation
#   alpha = r  vs alpha = 2*r  → Intensité

# 
#   HYPERPARAMÈTRES LoRA
# 
LORA_RANK    = 16   # Rang des matrices A et B
LORA_ALPHA   = 16   # Facteur d'échelle (alpha/r = 1 → effectif)
LORA_DROPOUT = 0    # Pas de dropout (recommandé Unsloth)
# 

model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_RANK,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    target_modules             = [
        "q_proj", "k_proj", "v_proj", "o_proj",   # Attention
        "gate_proj", "up_proj", "down_proj",        # MLP
    ],
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = GLOBAL_SEED,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params     = sum(p.numel() for p in model.parameters())
ratio            = 100 * trainable_params / total_params

print(f"{'=' * 60}")
print(f" Adaptateurs LoRA appliqués !")
print(f"{'─' * 60}")
print(f"   Rang (r)               : {LORA_RANK}")
print(f"   Alpha                  : {LORA_ALPHA}   (scaling = alpha/r = {LORA_ALPHA/LORA_RANK:.1f})")
print(f"   Modules ciblés         : q,k,v,o (attention) + gate,up,down (MLP)")
print(f"{'─' * 60}")
print(f"   Paramètres totaux      : {total_params:>14,}")
print(f"   Paramètres gelés       : {total_params - trainable_params:>14,}")
print(f"   Paramètres entraînables: {trainable_params:>14,}")
print(f"   Ratio entraîné         : {ratio:.2f}%")
print(f"{'=' * 60}")
print(f"\n On n'entraîne que {ratio:.2f}% du modèle — c'est la magie de LoRA !")


---
# Partie 4 : Chargement du Dataset (5 min)

## Dataset RGPD — 95 paires Q/R en français

Le dataset a été construit à partir du texte officiel du Règlement (UE) 2016/679 :

| Thème | Paires | Concepts couverts |
|---|---|---|
| Principes fondamentaux | ~15 | Licéité, minimisation, finalité, conservation, accountability |
| Bases légales (art. 6) | ~12 | Consentement, contrat, obligation légale, intérêt légitime |
| Droits des personnes | ~20 | Accès, rectification, effacement, portabilité, opposition, art. 22 |
| Obligations responsables | ~18 | DPO, AIPD, registre, sécurité, sous-traitants, violation |
| Données sensibles & transferts | ~15 | Art. 9, BCR, CCT, pays tiers, DPF |
| Sanctions & gouvernance | ~15 | CEPD, CNIL, amendes art. 83, recours |

Les réponses sont en **français académique** avec références aux articles et considérants exacts.

> **Prérequis :** Uploadez le fichier `dataset_rgpd_fr.json` dans Google Drive
> et ajustez `DATASET_PATH` ci-dessous.

### Format

```json
{"instruction": "Qu'est-ce que le RGPD et quand est-il entré en application ?",
 "response": "Le RGPD (Règlement Général sur la Protection des Données)..."}
```


In [ ]:
# Montage de Google Drive pour importer le dataset et sauvegarder les modèles


from google.colab import drive
drive.mount('/content/drive')

In [6]:
# Chargement du dataset Q/R RGPD depuis le fichier JSON


# Le dataset contient 95 paires question/réponse en français
# construites à partir du Règlement (UE) 2016/679 (RGPD).
#
# PRÉREQUIS : Uploadez 'dataset_rgpd_fr.json' dans Google Drive
# et ajustez DATASET_PATH ci-dessous.

import json
import os
from datasets import Dataset

# 
#  CHEMIN DU FICHIER — à adapter
# 
DATASET_PATH = "drive/MyDrive/content/dataset_rgpd_fr.json"
# 

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"Fichier introuvable : {DATASET_PATH}\n"
        "→ Uploadez 'dataset_rgpd_fr.json' dans Google Drive\n"
        "   et ajustez la variable DATASET_PATH."
    )

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    qa_pairs = json.load(f)

assert all("instruction" in p and "response" in p for p in qa_pairs), \
    "Format invalide : chaque paire doit avoir les clés 'instruction' et 'response'."

# Catégorisation thématique RGPD
THEMES = {
    "Droits des personnes" : ["droit d'accès", "effacement", "portabilité", "rectification",
                               "opposition", "limitation", "automatisée", "article 15",
                               "article 16", "article 17", "article 18", "article 20",
                               "article 21", "article 22"],
    "Bases légales"        : ["base légale", "consentement", "contrat", "obligation légale",
                               "intérêt légitime", "intérêts vitaux", "intérêt public",
                               "article 6"],
    "Obligations"          : ["DPO", "délégué", "AIPD", "registre", "sécurité",
                               "sous-traitant", "violation", "article 28", "article 30",
                               "article 32", "article 33", "article 35", "article 37"],
    "Transferts & sanctions": ["transfert", "pays tiers", "BCR", "CCT", "sanctions",
                                "amendes", "CNIL", "CEPD", "article 83"],
}

def categorize(instr):
    instr_lower = instr.lower()
    for theme, keywords in THEMES.items():
        if any(k.lower() in instr_lower for k in keywords):
            return theme
    return "Principes généraux"

sources = {}
for p in qa_pairs:
    src = categorize(p["instruction"])
    sources[src] = sources.get(src, 0) + 1

avg_len = sum(len(p["response"]) for p in qa_pairs) // len(qa_pairs)

print(f" Dataset RGPD chargé : {len(qa_pairs)} paires Q/R")
print(f"   Distribution thématique :")
for src, count in sorted(sources.items(), key=lambda x: -x[1]):
    print(f"     {src:<30} : {count} paires")
print(f"   Longueur moy. des réponses : {avg_len} caractères")
print(f"\n Exemple :")
print(f"{'─' * 60}")
print(f" Q : {qa_pairs[0]['instruction']}")
print(f" R : {qa_pairs[0]['response'][:200]}...")


In [7]:
# Formatage + Split train/validation

# Le SYSTEM_PROMPT spécialisé RGPD guide le modèle à répondre
# en français avec la terminologie juridique précise du règlement.

SYSTEM_PROMPT = (
    "Tu es un assistant expert en droit des données personnelles, "
    "spécialisé dans le Règlement Général sur la Protection des Données (RGPD), "
    "Règlement (UE) 2016/679 du Parlement européen et du Conseil du 27 avril 2016. "
    "Tu maîtrises parfaitement les 99 articles du RGPD, ses 173 considérants, "
    "les lignes directrices du CEPD, et la jurisprudence de la CJUE en matière de protection des données. "
    "Tu réponds TOUJOURS en français, de manière structurée, précise et pédagogique. "
    "Tu cites systématiquement les articles et considérants pertinents du RGPD "
    "et tu distingues clairement les obligations légales des bonnes pratiques recommandées."
)

# Récupérer le tokenizer pur (évite le bug ProcessorMixin de Qwen3.5 VLM)
_tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer

def format_example(example):
    """Convertit une paire instruction/réponse en conversation formatée."""
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]
    return {"text": _tok.apply_chat_template(
        messages, tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False
    )}

# Créer le dataset HuggingFace
full_dataset = Dataset.from_list(qa_pairs)

# Split train / validation (80% / 20%)
split         = full_dataset.train_test_split(test_size=0.2, seed=GLOBAL_SEED)
train_dataset = split["train"].map(format_example)
val_dataset   = split["test"].map(format_example)

print(f" Dataset total    : {len(full_dataset)} exemples")
print(f"   Train          : {len(train_dataset)} exemples (80%)")
print(f"   Validation     : {len(val_dataset)} exemples (20%)")
print(f"\n System prompt (extrait) :")
print(f"   '{SYSTEM_PROMPT[:120]}...'")
print(f"\n Aperçu du format de chat :")
print(f"{'─' * 60}")
print(train_dataset[0]["text"][:400])
print("...")


---
# Partie 5 : Entraînement avec SFT (30 min)

## Supervised Fine-Tuning (SFT)

Le SFT entraîne le modèle à **reproduire les réponses attendues** pour chaque instruction. C'est la forme la plus directe de fine-tuning supervisé.

## Hyperparamètres clés

| Paramètre | Rôle | Notre choix | Justification |
|---|---|---|---|
| **learning_rate** | Vitesse d'apprentissage | 2e-4 | Recommandé pour LoRA (QLoRA paper) |
| **num_train_epochs** | Passes sur le dataset | 3 | Compromis apprentissage/overfitting |
| **batch_size** | Exemples par step | 2 | Limité par VRAM |
| **gradient_accum** | Batch effectif | 4 → total 8 | Simule un grand batch |
| **warmup_steps** | Montée progressive du LR | 5 | Stabilise le début |
| **save_steps** | Checkpoint intermédiaire | 50 | Protection anti-déconnexion Colab |

> **Astuce Colab :** En cas de déconnexion, relancez avec `resume_from_checkpoint=True` dans `SFTConfig`.

In [11]:
# Test AVANT fine-tuning (baseline)

# Questions liées au RGPD — on teste si le modèle de base
# répond déjà en français avec la précision juridique souhaitée.

model.eval()  # Inférence standard HuggingFace — plus stable sur T4

# 
#  QUESTIONS DE TEST — couvrent 3 domaines clés du RGPD
# 
test_questions = [
    # Bases légales (art. 6)
    "Quelles sont les six bases légales permettant de traiter des données personnelles "
    "selon l'article 6 du RGPD ? Explique chacune avec un exemple concret.",

    # Droits des personnes (art. 17)
    "Qu'est-ce que le droit à l'effacement (droit à l'oubli) prévu à l'article 17 du RGPD ? "
    "Dans quels cas peut-il être exercé et quelles sont ses limites ?",

    # Obligations du responsable (art. 35)
    "Qu'est-ce qu'une Analyse d'Impact relative à la Protection des Données (AIPD) ? "
    "Quand est-elle obligatoire et que doit-elle contenir selon l'article 35 du RGPD ?",
]
# 

def generate_response(question, max_new_tokens=300):
    """Génère une réponse.
    use_cache=False évite le bug RoPE shape dans Unsloth Qwen3+float16.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    prompt_text = _tok.apply_chat_template(
        messages, tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = _tok(
        prompt_text, return_tensors="pt", return_attention_mask=True
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            temperature    = 0.7,
            do_sample      = True,
            pad_token_id   = _tok.eos_token_id,
            use_cache      = False,
        )
    return _tok.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )


print(" RÉPONSES AVANT FINE-TUNING (modèle de base)")
print("=" * 60)

reponses_avant = []
labels_questions = ["Bases légales (art. 6)", "Droit à l'effacement (art. 17)", "AIPD (art. 35)"]

for q, label in zip(test_questions, labels_questions):
    response = generate_response(q)
    reponses_avant.append(response)
    print(f"\n [{label}] {q[:80]}...")
    print(f"{'─' * 60}")
    print(response[:1000] + ("..." if len(response) > 1000 else ""))


In [12]:
# Entraînement SFT


import time
from trl import SFTTrainer, SFTConfig

# 
#  HYPERPARAMÈTRES — modifiables pour les exercices
# 
LEARNING_RATE    = 2e-4
NUM_EPOCHS       = 3
BATCH_SIZE       = 2
GRAD_ACCUM_STEPS = 4      # Batch effectif = 2 × 4 = 8
WARMUP_STEPS     = 5
# 

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    args          = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ_LENGTH,
        dataset_num_proc            = 2,
        packing                     = False,
        output_dir                  = "./outputs",
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM_STEPS,
        warmup_steps                = WARMUP_STEPS,
        num_train_epochs            = NUM_EPOCHS,
        learning_rate               = LEARNING_RATE,
        weight_decay                = 0.01,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 1,
        eval_strategy               = "epoch",
        save_strategy               = "steps",
        save_steps                  = 50,
        save_total_limit            = 2,
        optim                       = "adamw_torch",       # adamw_8bit évité (bitsandbytes non dispo sur T4)
        seed                        = GLOBAL_SEED,
        report_to                   = "none",
    ),
)

mem_before = torch.cuda.memory_allocated() / 1e9
print(f"{'=' * 60}")
print(f" Configuration de l'entraînement")
print(f"{'─' * 60}")
print(f"   Learning rate        : {LEARNING_RATE}")
print(f"   Époques              : {NUM_EPOCHS}")
print(f"   Batch effectif       : {BATCH_SIZE} × {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"   Train / Val          : {len(train_dataset)} / {len(val_dataset)} exemples")
print(f"   VRAM avant           : {mem_before:.2f} Go")
print(f"   Checkpoints          : tous les 50 steps → ./outputs/")
print(f"{'=' * 60}")
print("\n Lancement de l'entraînement...\n")

start_time    = time.time()
trainer_stats = trainer.train()
elapsed       = time.time() - start_time

mem_peak = torch.cuda.max_memory_allocated() / 1e9
print(f"\n{'=' * 60}")
print(f" Entraînement terminé !")
print(f"{'─' * 60}")
print(f"   Durée totale         : {elapsed:.0f} secondes ({elapsed/60:.1f} min)")
print(f"   Loss finale (train)  : {trainer_stats.metrics['train_loss']:.4f}")
print(f"   VRAM pic             : {mem_peak:.2f} Go / {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} Go")
print(f"{'=' * 60}")

In [13]:
# Courbe de loss (train vs validation)

# Interprétation :
#   train ↓ + val ↓         → apprentissage correct
#   train ↓ + val ↑ ou stagne → SURAPPRENTISSAGE (overfitting)
#   train ↓ + val ≈ train    → bonne généralisation

import matplotlib.pyplot as plt

log_history  = trainer.state.log_history
train_steps  = [x["step"] for x in log_history if "loss" in x and "eval_loss" not in x]
train_losses = [x["loss"] for x in log_history if "loss" in x and "eval_loss" not in x]
eval_steps   = [x["step"] for x in log_history if "eval_loss" in x]
eval_losses  = [x["eval_loss"] for x in log_history if "eval_loss" in x]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_steps, train_losses, color="#185FA5", linewidth=2,
        marker="o", markersize=4, label="Loss entraînement")
if eval_losses:
    ax.plot(eval_steps, eval_losses, color="#D85A30", linewidth=2,
            marker="s", markersize=6, linestyle="--", label="Loss validation")
ax.set_xlabel("Step", fontsize=12)
ax.set_ylabel("Loss (Cross-Entropy)", fontsize=12)
ax.set_title(
    f"Courbe de loss — {MODEL_NAME.split('/')[-1]}, r={LORA_RANK}, {NUM_EPOCHS} époques"
    f"\nDataset : RGPD — Règlement (UE) 2016/679",
    fontsize=12
)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig("loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n Interprétation :")
print(f"   Loss initiale (train) : {train_losses[0]:.4f}")
print(f"   Loss finale  (train)  : {train_losses[-1]:.4f}")
if eval_losses:
    gap = eval_losses[-1] - train_losses[-1]
    print(f"   Loss finale  (val)    : {eval_losses[-1]:.4f}")
    print(f"   Écart train/val       : {gap:+.4f}")
    if gap > 0.5:
        print("    Écart élevé → surapprentissage probable. Réduire NUM_EPOCHS ou LORA_RANK.")
    elif gap > 0.2:
        print("    Léger écart → entraînement correct. Surveiller avec plus d'époques.")
    else:
        print("    Bon équilibre train/val → généralisation satisfaisante !")

---
# Partie 6 : Évaluation — Avant vs Après (20 min)

Comparons les réponses sur les **mêmes questions RGPD**.

**Points à observer :**
- Le modèle cite-t-il maintenant les **articles exacts** du RGPD (art. 6, art. 17, art. 35) ?
- Les réponses sont-elles plus **structurées et précises** (numérotation, listes, conditions) ?
- Le modèle utilise-t-il la **terminologie juridique correcte** (responsable du traitement,
  personne concernée, base légale, AIPD, DPO, accountability) ?
- Y a-t-il des signes de **surapprentissage** (réponses copiées mot pour mot du dataset) ?
- Le modèle distingue-t-il correctement **obligations** (must) et **recommandations** (should) ?


In [14]:
# Test APRÈS fine-tuning


# FastLanguageModel.for_inference(model)  # Désactivé : bug RoPE shape
model.eval()

print(" RÉPONSES APRÈS FINE-TUNING")
print("=" * 60)

reponses_apres = []
for q, label in zip(test_questions, labels_questions):
    response = generate_response(q)
    reponses_apres.append(response)
    print(f"\n [{label}] {q[:80]}...")
    print(f"{'─' * 60}")
    print(response[:1000] + ("..." if len(response) > 1000 else ""))

In [15]:
# Comparaison côte à côte avant / après


print(" COMPARAISON AVANT / APRÈS FINE-TUNING")
print("=" * 70)

for i, (q, label) in enumerate(zip(test_questions, labels_questions)):
    print(f"\n{'━' * 70}")
    print(f" QUESTION {i+1} [{label}] : {q[:100]}...")
    print(f"{'━' * 70}")

    print(f"\n AVANT (modèle de base) :")
    print(f"{'─' * 35}")
    avant = reponses_avant[i]
    print(avant[:400] + ("..." if len(avant) > 400 else ""))

    print(f"\n APRÈS (fine-tuné sur le corpus RGPD) :")
    print(f"{'─' * 35}")
    apres = reponses_apres[i]
    print(apres[:400] + ("..." if len(apres) > 400 else ""))

print(f"\n{'━' * 70}")
print("\n Grille d'évaluation :")
print("   [1] La réponse est-elle entièrement en français ?")
print("   [2] Cite-t-elle des chiffres précis (ex: 99.6% réduction, 48 GB, NF4) ?")
print("   [3] Utilise-t-elle la terminologie exacte des papiers ?")
print("   [4] La réponse est-elle structurée (paragraphes, liste) ?")
print("   [5] Y a-t-il des signes de surapprentissage (copie verbatim du chunk) ?")

In [16]:
# Test de généralisation

# Ces questions ne figurent PAS dans le dataset d'entraînement.
# Si le modèle répond bien → il a appris le STYLE et le DOMAINE,
# pas juste mémorisé des phrases.

questions_generalisation = [
    # Application pratique cross-concept
    "Une startup de 50 personnes collecte des emails et adresses IP pour envoyer une newsletter. "
    "Quelles sont ses obligations RGPD prioritaires (base légale, information, conservation) ?",

    # Distinction nuancée
    "Quelle est la différence entre pseudonymisation et anonymisation au sens du RGPD, "
    "et pourquoi cette distinction est-elle importante pour le champ d'application du règlement ?",

    # Scénario de violation
    "Une entreprise subit une cyberattaque exposant 50 000 dossiers clients (nom, email, IBAN). "
    "Quelles sont ses obligations de notification au titre des articles 33 et 34 du RGPD "
    "et dans quels délais ?",
]

print(" TEST DE GÉNÉRALISATION (questions hors dataset)")
print("=" * 60)

for q in questions_generalisation:
    response = generate_response(q, max_new_tokens=350)
    print(f"\n {q[:100]}...")
    print(f"{'─' * 60}")
    print(response[:500] + ("..." if len(response) > 500 else ""))

print(f"\n{'━' * 60}")
print(" Interprétation :")
print("   → Références aux articles RGPD = le modèle a appris le DOMAINE")
print("   → Structure juridique (conditions, exceptions) = il a appris le STYLE")
print("   → Confusions ou omissions = fine-tuning insuffisant ou overfitting")


---
# Partie 7 : Sauvegarde et Export du modèle (10 min)

| Méthode | Taille | Usage |
|---|---|---|
| **Adaptateurs LoRA seuls** | ~100-200 Mo | Léger, nécessite le modèle de base pour recharger |
| **Modèle fusionné (merged)** | Taille complète | Autonome, prêt à déployer |
| **Format GGUF** | Variable | Pour llama.cpp, Ollama, LM Studio |

Nous sauvegardons les **adaptateurs LoRA uniquement** — méthode légère et pratique.

In [17]:
# Sauvegarde des adaptateurs LoRA sur Google Drive


import os, shutil

DRIVE_OUTPUT_DIR = 'drive/MyDrive/content/model'
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

print(f" Sauvegarde dans : {DRIVE_OUTPUT_DIR}")

model.save_pretrained(DRIVE_OUTPUT_DIR)
tokenizer.save_pretrained(DRIVE_OUTPUT_DIR)

fichiers      = os.listdir(DRIVE_OUTPUT_DIR)
taille_totale = sum(
    os.path.getsize(os.path.join(DRIVE_OUTPUT_DIR, f)) for f in fichiers
) / 1e6

print(f"\n Sauvegarde terminée !")
print(f"   Dossier  : {DRIVE_OUTPUT_DIR}")
print(f"   Fichiers : {fichiers}")
print(f"   Taille   : {taille_totale:.1f} Mo")
print(f"\n Seuls les adaptateurs LoRA sont sauvegardés ({taille_totale:.0f} Mo)")
print(f"   et non le modèle complet !")
print(f"\n Pour recharger l'adapter :")
print(f"   model, tok = FastLanguageModel.from_pretrained('{MODEL_NAME}', ...)")
print(f"   model = PeftModel.from_pretrained(model, '{DRIVE_OUTPUT_DIR}')")

if os.path.exists('loss_curve.png'):
    shutil.copy('loss_curve.png', DRIVE_OUTPUT_DIR)
    print(f"\n   Courbe de loss copiée dans {DRIVE_OUTPUT_DIR}/")


In [18]:
# CELLULE 15b — Export GGUF pour Ollama (inférence locale)

# Cette cellule fusionne les adaptateurs LoRA dans le modèle
# de base et exporte en format GGUF quantifié.
#
# PRÉREQUIS : Avoir exécuté la cellule 14 (sauvegarde LoRA)
#
# APRÈS L'EXPORT :
#   1. Téléchargez le fichier .gguf depuis le panneau fichiers
#   2. Installez Ollama sur votre machine : https://ollama.com
#   3. Suivez les instructions affichées en fin de cellule

import os

# ═══════════════════════════════════════════════════════════════
#  PARAMÈTRES D'EXPORT
# ═══════════════════════════════════════════════════════════════
GGUF_DIR            = "drive/MyDrive/content/model-gguf"
QUANTIZATION_METHOD = "q4_k_m"   # Meilleur compromis taille/qualité
                                  # Autres options :
                                  # "q2_k"   → très léger, qualité réduite
                                  # "q4_k_m" → recommandé ← notre choix
                                  # "q5_k_m" → meilleure qualité, plus lourd
                                  # "q8_0"   → quasi-lossless, très lourd
# ═══════════════════════════════════════════════════════════════

print(f" Export GGUF en cours...")
print(f"   Méthode de quantification : {QUANTIZATION_METHOD}")
print(f"   Dossier de sortie         : {GGUF_DIR}")
print(f"   (Cette opération prend 5-15 minutes selon le modèle)\n")

# Fusion des adaptateurs + export GGUF
model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method = QUANTIZATION_METHOD,
)

# Unsloth crée un sous-dossier GGUF_DIR + "_gguf"
GGUF_ACTUAL_DIR = GGUF_DIR + "_gguf"
gguf_files = [f for f in os.listdir(GGUF_ACTUAL_DIR) if f.endswith(".gguf")] if os.path.exists(GGUF_ACTUAL_DIR) else []
if not gguf_files:  # fallback si dans GGUF_DIR directement
    gguf_files = [f for f in os.listdir(GGUF_DIR) if f.endswith(".gguf")]
    GGUF_ACTUAL_DIR = GGUF_DIR
gguf_path  = os.path.join(GGUF_ACTUAL_DIR, gguf_files[0]) if gguf_files else None
gguf_size  = os.path.getsize(gguf_path) / 1e9 if gguf_path else 0

print(f"\n Export terminé !")
print(f"   Fichier GGUF : {gguf_path}")
print(f"   Taille       : {gguf_size:.1f} Go")

# ── Générer le Modelfile pour Ollama ──────────────────────────
MODELFILE_PATH = os.path.join(GGUF_ACTUAL_DIR, "Modelfile")
MODEL_OLLAMA   = "peft-rgpd-fr"

MODELFILE_CONTENT = f"""FROM ./{gguf_files[0] if gguf_files else 'model.gguf'}

SYSTEM \"\"\"{SYSTEM_PROMPT}\"\"\"

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER num_ctx 2048
PARAMETER stop "<|im_end|>"
"""

with open(MODELFILE_PATH, "w") as f:
    f.write(MODELFILE_CONTENT)

print(f"   Modelfile    : {MODELFILE_PATH}")


---
## Instructions : utiliser le modèle en local avec Ollama

### Prérequis
- Avoir terminé l'export GGUF (cellule ci-dessus)
- Avoir installé [Ollama](https://ollama.com/download)

### Étape 1 — Télécharger les 2 fichiers depuis Colab

Dans le panneau fichiers (icône dossier à gauche), clic droit → **Télécharger** :

```
drive/MyDrive/content/model-gguf_gguf/model.gguf
drive/MyDrive/content/model-gguf_gguf/Modelfile
```

### Étape 2 — Créer le modèle Ollama

Placez les 2 fichiers dans le même dossier, puis dans un terminal :

```bash
cd /chemin/vers/dossier/model-gguf_gguf
ollama create peft-rgpd-fr -f Modelfile
```

### Étape 3 — Lancer le modèle

```bash
ollama run peft-rgpd-fr
```

### Étape 4 — Questions de test recommandées

```
>>> Quelles sont les six bases légales de l'article 6 du RGPD ?
>>> Qu'est-ce que le droit à l'effacement prévu à l'article 17 ?
>>> Quand une AIPD est-elle obligatoire selon l'article 35 ?
>>> Quelle est la différence entre pseudonymisation et anonymisation ?
>>> Une startup collecte des emails pour une newsletter : quelles obligations RGPD ?
```

### Commandes utiles

```bash
ollama list                  # voir tous les modèles installés
ollama show peft-rgpd-fr     # infos du modèle
ollama rm peft-rgpd-fr       # supprimer le modèle
```

### API REST (optionnel)

```bash
curl http://localhost:11434/api/generate -d '{
  "model": "peft-rgpd-fr",
  "prompt": "Qu'est-ce que le consentement valide selon le RGPD ?",
  "stream": false
}'
```

> **Modèle de base :** `unsloth/Qwen3-4B` fine-tuné sur 95 paires Q/R extraites du RGPD (Règlement (UE) 2016/679)  
> **Quantification :** Q4_K_M (~2.5 Go)  
> **Format :** GGUF pour Ollama


In [ ]:

# (Optionnel) Push vers Hugging Face Hub


# from huggingface_hub import login
# from getpass import getpass
#
# HF_TOKEN = getpass(" Token HuggingFace (write) : ")
# login(token=HF_TOKEN)
#
# HF_REPO = f"votre-username/lora-{MODEL_SHORT}-rgpd-fr"
# model.push_to_hub(HF_REPO)
# tokenizer.push_to_hub(HF_REPO)
# print(f" Modèle publié : https://huggingface.co/{HF_REPO}")

print(" Décommentez cette cellule pour publier sur Hugging Face Hub.")


---
# Partie 8 : Exercices d'approfondissement

## Exercice 1 : Impact du rang LoRA

Remontez à la **Cellule 5** et testez :

| Valeur | Params entraînables | Comportement attendu |
|---|---|---|
| `r = 4` | ~8M | Adaptation minimale, terminologie juridique moins précise |
| `r = 16` | ~30M | Notre choix par défaut |
| `r = 64` | ~120M | Risque de surapprentissage sur 76 exemples d'entraînement |

> Comparez les scores de loss finale et la précision des réponses sur les 3 questions RGPD.

## Exercice 2 : Overfitting délibéré

Mettez `NUM_EPOCHS = 15` dans la Cellule 10. La courbe de loss devrait montrer :
- Train loss → continue à descendre
- Val loss → commence à remonter après le point optimal

Quel est le nombre optimal d'époques pour votre dataset de 76 exemples d'entraînement ?

## Exercice 3 : QLoRA vs LoRA float16 — comparaison empirique

Sur T4 (15 Go VRAM), modifiez la Cellule 4 :
```python
load_in_4bit  = True   # QLoRA activé
load_in_16bit = False
```
Lancez le même TP avec QLoRA. Les réponses sont-elles aussi précises sur les articles du RGPD ?
La quantification NF4 dégrade-t-elle la compréhension des concepts juridiques ?

## Exercice 4 : Questions de généralisation avancées

Ajoutez vos propres questions dans la Cellule 13 et évaluez la qualité des réponses :
```python
# Exemples à tester :
"Qu'est-ce que le mécanisme du guichet unique (one-stop-shop) dans le RGPD ?"
"Quelle est la différence entre un responsable du traitement et un co-responsable selon l'article 26 ?"
"Dans quels cas une entreprise hors UE est-elle soumise au RGPD (article 3) ?"
```

## Exercice 5 : Enrichissement du dataset

Ajoutez 20 nouvelles paires Q/R à `dataset_rgpd_fr.json` couvrant des articles non encore représentés :
- Article 46 (garanties appropriées pour les transferts)
- Article 58 (pouvoirs des autorités de contrôle)
- Article 79 (droit à un recours juridictionnel effectif)

Relancez l'entraînement et observez si la loss finale est plus basse.


In [ ]:
# ============================================================
# CELLULE 16 — Espace libre pour vos expérimentations
# ============================================================

# Exemple : tester une question personnalisée sur le RGPD
# ma_question = "Qu'est-ce que le principe d'accountability dans le RGPD ?"
# print(generate_response(ma_question, max_new_tokens=400))

# --- VOTRE CODE ICI ---



# --- FIN DE VOTRE CODE ---


---
# Conclusion et points clés

## Ce que vous avez appris

| Concept | Ce que vous maîtrisez |
|---|---|
| **LoRA (Hu et al., 2021)** | $h = W_0 x + \\frac{\\alpha}{r} BAx$, rang intrinsèque bas, ~1.6% params |
| **QLoRA (Dettmers et al., 2023)** | NF4 + Double Quantization + Paged Optimizers |
| **Fine-tuning domaine juridique** | Spécialiser un LLM sur un corpus réglementaire en français |
| **SYSTEM_PROMPT spécialisé** | Forcer la terminologie juridique précise (articles, considérants) |
| **Split train/val** | Surveiller le surapprentissage sur petit dataset (95 paires) |
| **Courbe de loss** | Interpréter train vs val loss, détecter l'overfitting |
| **Export GGUF / Ollama** | Déployer un assistant RGPD en local |


